# DACKAR — Outage Activity Analysis Pipeline Demo

**What is this?**  DACKAR is an AI-assisted decision-support system for nuclear plant outage management.
When an unexpected activity is discovered during an outage, DACKAR automatically analyses its history,
temporal context, and schedule impact, then recommends the safest and most efficient course of action.

**Two demo scenarios** exercise every stage of the pipeline and show two opposite outcomes:

| # | Scenario | Emergence Type | CP Impact | Regulatory Constraint | Expected Decision |
|---|----------|---------------|-----------|----------------------|-------------------|
| 1 | RCP Train-A Mechanical Seal Leak | `regulatory_driven` | CRITICAL — 48 h drag | TS 3.4.6 (defer prohibited) | **🔴 ESCALATE** |
| 2 | Snubber Inspection Scope Expansion | `scope_expansion` | NON-CRITICAL — 0 h drag | None | **🟢 PROCEED** |

---

**Stage execution summary:**

| Stage | Name | Mode | Description |
|-------|------|------|-------------|
| A | Activity Intake & Classification | Live | Classifies the activity, assesses data quality |
| B | KG Timeline Builder | Live (stub backend) | Retrieves prior events from the Knowledge Graph |
| C | Temporal Event Chain | Live | Allen interval algebra — was this event caused by something earlier? |
| D | Historical Analog Retriever | Live (stub backend) | Finds similar past activities; fits a duration distribution |
| E | Schedule Impact Assessment | Stub | Computes CP float impact (requires LOGOS CPM engine in production) |
| F | Insertion Option Generator | Live | Generates and risk-scores all viable insertion options |
| G | Recommendation Synthesiser | Live | Produces the final decision + evidence chain |

> **Stub backends** replace Neo4j (KG) and a vector embedding server (retrieval) with
> in-memory fixtures — so the demo runs anywhere with no external services.


## 0 · Setup

This cell imports Python libraries and the two demo scenario definitions.
It also configures the Matplotlib plot style used throughout the notebook.

**For developers:** `demo_scenarios.py` in this folder contains all stub backends
(`_StubKGDriver`, `_StubRetrievalIndex`) and the two pre-built scenario dictionaries.
No external services (Neo4j, embedding server, LOGOS) are needed.


In [ ]:
import sys
import warnings
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.ticker import MaxNLocator
import numpy as np

# Suppress noisy library warnings and debug log spam during the demo
warnings.filterwarnings('ignore')
logging.disable(logging.WARNING)

# Add the outage package and demo folder to sys.path.
# Notebook lives two levels below outage/:
#   demos/unexpected_act_workflow_1/ → demos/ → outage/
DEMO_DIR   = Path().resolve()              # .../demos/unexpected_act_workflow_1/
OUTAGE_DIR = DEMO_DIR.parent.parent        # .../outage/
if str(OUTAGE_DIR) not in sys.path:
    sys.path.insert(0, str(OUTAGE_DIR))
if str(DEMO_DIR) not in sys.path:
    sys.path.insert(0, str(DEMO_DIR))

# Import the two pre-built scenario dicts and the orchestrator function
from demo_scenarios import run_pipeline, SCENARIO_RCP_SEAL, SCENARIO_SNUBBER_EXT

# ── Plot style ────────────────────────────────────────────────────────────────
# These settings apply globally to every matplotlib figure in the notebook
plt.rcParams.update({
    'figure.dpi': 110,              # crisp rendering for screen and PDF
    'axes.spines.top': False,       # remove top/right borders for a clean look
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 12,
    'axes.labelsize': 10,
})

# Shared colour palette — keeps all plots visually consistent.
# Keys map to decision outcomes (escalate/proceed/defer) and confidence tiers.
PALETTE = {
    'escalate':       '#D7263D',   # red
    'proceed':        '#06A77D',   # teal-green
    'defer':          '#F4A261',   # amber
    'monitor':        '#A8DADC',   # light blue
    'blocked':        '#E76F51',   # orange-red (infeasible but exists)
    'infeasible':     '#CCCCCC',   # light grey
    'data_supported': '#264653',   # dark teal  (≥5 analogues)
    'sme_informed':   '#2A9D8F',   # medium teal (1–4 analogues)
    'low_confidence': '#E9C46A',   # gold        (0 analogues)
    'stage_live':     '#264653',   # dark teal   (stage runs real logic)
    'stage_stub':     '#94A3B8',   # slate grey  (stage uses pre-built data)
    'stage_text':     '#FFFFFF',   # white text on stage tiles
}

print('Setup complete.')


## 1 · Run the Pipeline

The cell below runs both scenarios end-to-end through the DACKAR pipeline.
Each call to `run_pipeline()` executes stages A → B → C → D → E → F → G in sequence
and returns a dictionary containing every stage's output artifact.

**For managers:** This is the single entry point — one function call per scenario.
The pipeline typically completes in under 200 ms (stub backends; production with
Neo4j and embedding server will be in the 1–3 s range per scenario).


In [ ]:
import time

# --- Run Scenario 1: RCP Seal Leak (expected: ESCALATE) ---
t0 = time.perf_counter()
r1 = run_pipeline(SCENARIO_RCP_SEAL)       # dict with keys: scenario_label, run_id,
t1 = time.perf_counter()                   # intake, timeline, temporal, analogs,
                                           # schedule, options, recommendation

# --- Run Scenario 2: Snubber Scope Expansion (expected: PROCEED) ---
t2 = time.perf_counter()
r2 = run_pipeline(SCENARIO_SNUBBER_EXT)
t3 = time.perf_counter()

# Print a quick summary — scenario_label already contains the scenario name,
# so we do NOT add "Scenario N —" prefix here to avoid duplication.
print(r1['scenario_label'])
print(f"  Decision : {r1['recommendation']['decision_status']}")
print(f"  Runtime  : {(t1-t0)*1000:.0f} ms")
print()
print(r2['scenario_label'])
print(f"  Decision : {r2['recommendation']['decision_status']}")
print(f"  Runtime  : {(t3-t2)*1000:.0f} ms")


## 2 · Pipeline Architecture

The diagram below shows how the seven stages are connected.
Stages in **dark teal** run real production logic; **grey** stages use pre-built stub data.

Arrows show data flow: each stage's output JSON becomes the next stage's input.
Stages B and D run concurrently (they are independent of each other) in the
production orchestrator, though in this demo they run sequentially for simplicity.


In [ ]:
def draw_pipeline_architecture():
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.set_xlim(0, 14)
    ax.set_ylim(0, 4)
    ax.axis('off')
    ax.set_title('DACKAR Outage Activity Analysis — Stage Pipeline', fontsize=13, fontweight='bold', pad=12)

    stages = [
        # (x_center, y_center, label, sublabel, is_stub)
        (1.0,  2.0, 'A', 'Activity\nIntake',         False),
        (3.0,  3.0, 'B', 'KG\nTimeline',             False),   # parallel upper
        (3.0,  1.0, 'D', 'Historical\nAnalogs',      False),   # parallel lower
        (5.5,  2.0, 'C', 'Temporal\nChain',          False),
        (8.0,  2.0, 'E', 'Schedule\nImpact',         True),    # stub
        (10.5, 2.0, 'F', 'Option\nGeneration',       False),
        (13.0, 2.0, 'G', 'Recommendation\nSynthesis',False),
    ]

    BOX_W, BOX_H = 1.6, 0.9
    boxes = {}
    for (cx, cy, lbl, sub, stub) in stages:
        color = PALETTE['stage_stub'] if stub else PALETTE['stage_live']
        rect = mpatches.FancyBboxPatch(
            (cx - BOX_W/2, cy - BOX_H/2), BOX_W, BOX_H,
            boxstyle='round,pad=0.08', linewidth=1.5,
            edgecolor='white', facecolor=color, zorder=3
        )
        ax.add_patch(rect)
        ax.text(cx, cy + 0.10, lbl, ha='center', va='center',
                fontsize=15, fontweight='bold', color='white', zorder=4)
        ax.text(cx, cy - 0.22, sub, ha='center', va='center',
                fontsize=7.0, color='white', zorder=4, linespacing=1.3)
        boxes[lbl] = (cx, cy)

    # Draw arrows
    def arrow(x1, y1, x2, y2, color='#475569', label=''):
        ax.annotate('', xy=(x2 - BOX_W/2 - 0.05, y2),
                    xytext=(x1 + BOX_W/2 + 0.05, y1),
                    arrowprops=dict(arrowstyle='->', color=color, lw=1.5))
        if label:
            mx, my = (x1 + x2) / 2, (y1 + y2) / 2
            ax.text(mx, my + 0.12, label, ha='center', fontsize=7, color=color)

    # A → B (upper)
    ax.annotate('', xy=(boxes['B'][0] - BOX_W/2 - 0.05, boxes['B'][1]),
                xytext=(boxes['A'][0] + BOX_W/2 + 0.05, boxes['A'][1]),
                arrowprops=dict(arrowstyle='->', color='#475569', lw=1.5,
                                connectionstyle='arc3,rad=-0.35'))
    # A → D (lower)
    ax.annotate('', xy=(boxes['D'][0] - BOX_W/2 - 0.05, boxes['D'][1]),
                xytext=(boxes['A'][0] + BOX_W/2 + 0.05, boxes['A'][1]),
                arrowprops=dict(arrowstyle='->', color='#475569', lw=1.5,
                                connectionstyle='arc3,rad=0.35'))
    # B → C
    ax.annotate('', xy=(boxes['C'][0] - BOX_W/2 - 0.05, boxes['C'][1]),
                xytext=(boxes['B'][0] + BOX_W/2 + 0.05, boxes['B'][1]),
                arrowprops=dict(arrowstyle='->', color='#475569', lw=1.5,
                                connectionstyle='arc3,rad=0.30'))
    # D → C (feeds from below)
    ax.annotate('', xy=(boxes['C'][0] - BOX_W/2 - 0.05, boxes['C'][1]),
                xytext=(boxes['D'][0] + BOX_W/2 + 0.05, boxes['D'][1]),
                arrowprops=dict(arrowstyle='->', color='#475569', lw=1.5,
                                connectionstyle='arc3,rad=-0.30'))
    # C → E
    arrow(boxes['C'][0], boxes['C'][1], boxes['E'][0], boxes['E'][1])
    # E → F
    arrow(boxes['E'][0], boxes['E'][1], boxes['F'][0], boxes['F'][1])
    # F → G
    arrow(boxes['F'][0], boxes['F'][1], boxes['G'][0], boxes['G'][1])

    # Legend
    leg = [
        mpatches.Patch(color=PALETTE['stage_live'], label='Real production logic'),
        mpatches.Patch(color=PALETTE['stage_stub'], label='Pre-built stub (LOGOS CPM)'),
    ]
    ax.legend(handles=leg, loc='lower right', fontsize=8, framealpha=0.7)

    plt.tight_layout()
    plt.show()

draw_pipeline_architecture()

---
## 3 · Scenario 1 — RCP Train-A Seal Leak  🔴 ESCALATE

**What happened?** During the outage, a mechanical seal leak was detected on RCP Train-A.
This is a **safety-related** component covered by Technical Specification 3.4.6,
which **prohibits deferral** — the activity must be addressed, but inserting it now
conflicts with crew availability and adds 48 hours of critical-path drag.

**Why ESCALATE?** Every insertion option is either infeasible or contradicted by
regulatory constraints. The pipeline correctly flags this for senior engineering
and plant management review rather than making an autonomous decision.

The cells below walk through each stage's contribution to this conclusion.


### 3.1 · Stage A — Activity Intake & Classification

**What Stage A does:**
Stage A is the pipeline entry point. It takes the raw activity description, assigns a
unique activity ID, classifies the *emergence type* (e.g., `regulatory_driven`,
`scope_expansion`, `equipment_failure`), scores data quality, and flags whether
regulatory constraints are present.

**Key outputs to look for:**
- `emergence_type` and its confidence score
- `data_quality_score` (0–1): how complete and reliable the input data is
- `has_regulatory_constraint`: if True, Stage F will be more restrictive


In [ ]:
# Retrieve the Stage A output artifact for Scenario 1
intake = r1['intake']

print('=== STAGE A — Activity Intake ===')
print(f"  Activity ID          : {intake['activity_id']}")

# emergence_type: how this activity entered the outage scope
# confidence: ML classifier score (1.0 = certain, <0.7 = uncertain)
print(f"  Emergence type       : {intake['emergence_type']}  (confidence {intake['emergence_type_confidence']:.0%})")

# has_regulatory_constraint: True means Stage F cannot mark deferral as feasible
print(f"  Has regulatory const : {intake['has_regulatory_constraint']}")

# data_quality_score: affects confidence tier propagated to Stages D, F, and G
print(f"  Data quality score   : {intake['data_quality_score']:.2f}")


In [ ]:
def plot_stage_a_summary(intake, title='Stage A Summary'):
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
    fig.suptitle(title, fontsize=12, fontweight='bold')

    # ── Panel 1: Data quality gauge ───────────────────────────────────────────
    ax = axes[0]
    dq = intake['data_quality_score']
    abbr = intake['unknown_abbreviation_rate']
    metrics = ['Data Quality', 'Abbr Clarity']
    values  = [dq, 1 - abbr]
    colors  = [PALETTE['proceed'] if v >= 0.6 else PALETTE['defer'] for v in values]
    bars = ax.barh(metrics, values, color=colors, edgecolor='white', height=0.45)
    ax.set_xlim(0, 1)
    ax.axvline(0.6, ls='--', color='#888', lw=1, label='threshold 0.6')
    ax.set_title('Input Quality Scores')
    ax.set_xlabel('Score [0–1]')
    for bar, v in zip(bars, values):
        ax.text(min(v + 0.02, 0.95), bar.get_y() + bar.get_height()/2,
                f'{v:.2f}', va='center', fontsize=9)
    ax.legend(fontsize=7)

    # ── Panel 2: Entity types ─────────────────────────────────────────────────
    ax = axes[1]
    entities = intake['extracted_entities']
    from collections import Counter
    type_counts = Counter(e['entity_type'] for e in entities)
    if type_counts:
        types, counts = zip(*sorted(type_counts.items(), key=lambda x: -x[1]))
        ax.barh(types, counts, color=PALETTE['stage_live'], edgecolor='white', height=0.5)
        ax.set_xlabel('Count')
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_title(f'Extracted Entities ({len(entities)})')

    # ── Panel 3: Regulatory drivers ───────────────────────────────────────────
    ax = axes[2]
    drivers = intake['regulatory_drivers']
    if drivers:
        labels = [d['driver_type'].replace('_', '\n') for d in drivers]
        colors_d = [PALETTE['escalate'] if d['defer_prohibited'] else PALETTE['defer']
                    for d in drivers]
        ax.barh(labels, [1]*len(drivers), color=colors_d, edgecolor='white', height=0.5)
        ax.set_xlim(0, 1.5)
        ax.set_xticks([])
        ax.set_title(f'Regulatory Drivers ({len(drivers)})')
        red_p  = mpatches.Patch(color=PALETTE['escalate'], label='defer_prohibited')
        yell_p = mpatches.Patch(color=PALETTE['defer'],    label='defer_allowed')
        ax.legend(handles=[red_p, yell_p], fontsize=7, loc='lower right')
    else:
        ax.text(0.5, 0.5, 'No regulatory\nconstraints detected',
                ha='center', va='center', fontsize=10, color='#888')
        ax.set_title('Regulatory Drivers (0)')
        ax.axis('off')

    plt.tight_layout()
    plt.show()

plot_stage_a_summary(r1['intake'], title='Stage A — RCP Seal Scenario')

### 3.2 · Stage C — Temporal Event Chain (Allen Interval Algebra)

**What Stage C does:**
Stage C asks: *"Did anything happen before this activity that could explain it?"*

It retrieves the component's event history from the Knowledge Graph (Stage B) and
classifies each historical event using **Allen interval algebra** — a mathematical
framework for describing how two time intervals relate to each other:

| Allen Relation | Meaning | Causal Score |
|---------------|---------|-------------|
| OVERLAPS | Prior event was still active when this activity began | 0.90 |
| CONTAINS | Long-running event that spans the entire activity window | 0.85 |
| PRECEDES | Prior event ended before this activity started (classic lead-time) | 0.75 |
| SIMULTANEOUS | Concurrent — possible common-cause scenario | 0.50 |
| DURING | Started *after* the activity — likely a symptom, not a cause | 0.30 |
| FOLLOWS | Temporal contradiction — flagged for analyst review | 0.10 |

**Key output to look for:** `causal_posture` in the summary
(`supported` / `partial` / `contradicted` / `insufficient_data`)


In [ ]:
def plot_allen_timeline(temporal, title='Stage C — Temporal Event Chain'):
    import datetime
    chain_links = temporal.get('chain_links', [])
    activity_interval = temporal.get('emergent_activity_interval', {})
    summary = temporal.get('summary', {})

    if not chain_links:
        print('No chain links to visualise.')
        return

    ALLEN_COLORS = {
        'precedes':     '#2A9D8F',
        'overlaps':     '#E76F51',
        'contains':     '#D7263D',
        'during':       '#A8DADC',
        'follows':      '#E9C46A',
        'simultaneous': '#264653',
        'unknown':      '#CCCCCC',
    }

    fig, ax = plt.subplots(figsize=(13, max(3, len(chain_links) * 0.9 + 2)))
    ax.set_title(title, fontsize=12, fontweight='bold')

    # ── Parse emergent activity timestamps to offsets (days before detection) ─
    def _days_ago(ts_str):
        try:
            from datetime import datetime, timezone
            ts = datetime.fromisoformat(ts_str)
            if ts.tzinfo is None:
                ts = ts.replace(tzinfo=timezone.utc)
            act_ts = datetime.fromisoformat(activity_interval.get('start', ts_str))
            if act_ts.tzinfo is None:
                act_ts = act_ts.replace(tzinfo=timezone.utc)
            return (ts - act_ts).days
        except Exception:
            return 0

    y_positions = list(range(len(chain_links)))
    labels = []

    for i, link in enumerate(chain_links):
        y = i
        x_event = _days_ago(link['event_timestamp'])
        rel = link['allen_relation']
        color = ALLEN_COLORS.get(rel, '#888')
        score = link.get('relation_score', 0)
        conf  = link.get('confidence', 0)

        # Event dot
        ax.scatter(x_event, y, s=120, color=color, zorder=5, edgecolors='white', linewidths=1)

        # Arrow from event toward activity (x=0)
        ax.annotate('', xy=(0, y), xytext=(x_event + (0.2 if x_event < 0 else -0.2), y),
                    arrowprops=dict(arrowstyle='->', color=color, lw=1.5, alpha=0.7))

        # Label
        label = f"{link['event_type']}  |  {rel.upper()}  |  score={score:.2f}  conf={conf:.2f}"
        labels.append(label)

    # Emergent activity vertical line
    ax.axvline(0, color=PALETTE['escalate'], lw=2.5, ls='-', label='Emergent activity detected', zorder=4)

    ax.set_yticks(y_positions)
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('Days relative to detection (negative = before detection)')
    ax.invert_yaxis()

    # Allen relation legend
    legend_patches = [mpatches.Patch(color=c, label=r.upper())
                      for r, c in ALLEN_COLORS.items()]
    ax.legend(handles=legend_patches, fontsize=7, loc='lower right',
              title='Allen Relation', title_fontsize=7, ncol=2)

    # Summary banner
    posture  = summary.get('causal_posture', '?')
    n_strong = summary.get('strong_link_count', 0)
    ax.set_title(
        f"{title}\n"
        f"Causal posture: {posture.upper()}  |  "
        f"Strong links: {n_strong}  |  "
        f"Temporal contradiction: {summary.get('has_temporal_contradiction', False)}",
        fontsize=10
    )

    plt.tight_layout()
    plt.show()

plot_allen_timeline(r1['temporal'], title='Stage C — RCP Seal Scenario (Allen Relations)')

### 3.3 · Stage D — Historical Analog Distribution

**What Stage D does:**
Stage D searches the historical database for past activities that are similar
to the current one ("analogues") and fits a statistical duration distribution
to their actual completion times.

This answers: *"How long does this type of work typically take, based on history?"*

**Confidence tiers** (driven by analogue count):

| Tier | Analogues Found | Meaning |
|------|----------------|---------|
| `data_supported` | ≥ 5 | Strong historical evidence; high confidence |
| `sme_informed` | 1–4 | Some history; SME judgment still needed |
| `low_confidence` | 0 | No history found; treat estimates cautiously |

**Key outputs to look for:** `confidence_tier`, `p50_hours` (median), `p80_hours` (planning estimate)


In [ ]:
def plot_analog_distribution(analogs_result, title='Stage D — Analog Duration Distribution'):
    analogs = analogs_result.get('analogs', [])
    dist    = analogs_result.get('duration_distribution', {})
    summary = analogs_result.get('retrieval_summary', {})

    durations = [a['actual_duration_hours'] for a in analogs
                 if a.get('actual_duration_hours') is not None]

    if not durations:
        print('No analog durations available.')
        return

    tier   = dist.get('confidence_tier', 'low_confidence')
    p50    = dist.get('p50_hours')
    p80    = dist.get('p80_hours')
    p90    = dist.get('p90_hours')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(title, fontsize=12, fontweight='bold')

    # ── Panel 1: Histogram ────────────────────────────────────────────────────
    tier_color = PALETTE.get(tier, '#888')
    ax1.hist(durations, bins=max(3, len(durations)//2 + 1),
             color=tier_color, edgecolor='white', alpha=0.85, rwidth=0.8)

    # Percentile lines
    if p50: ax1.axvline(p50, color='#F4A261', lw=2, ls='--', label=f'p50 = {p50:.0f} h')
    if p80: ax1.axvline(p80, color='#E76F51', lw=2, ls='--', label=f'p80 = {p80:.0f} h')
    if p90: ax1.axvline(p90, color='#D7263D', lw=2, ls='--', label=f'p90 = {p90:.0f} h')

    ax1.set_xlabel('Actual Duration (hours)')
    ax1.set_ylabel('Count')
    ax1.set_title(f'Duration Histogram (n={len(durations)}, tier={tier})')
    ax1.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax1.legend(fontsize=8)

    # ── Panel 2: Per-analog dot plot ──────────────────────────────────────────
    sorted_a = sorted(analogs, key=lambda x: x.get('actual_duration_hours') or 0)
    ys = list(range(len(sorted_a)))
    xs = [a.get('actual_duration_hours') or 0 for a in sorted_a]
    labels = [f"{a.get('outage_id', '?')} — {a.get('description', '')[:40]}" for a in sorted_a]

    ax2.scatter(xs, ys, color=tier_color, s=80, zorder=5, edgecolors='white')
    if p50: ax2.axvline(p50, color='#F4A261', lw=2, ls='--', alpha=0.8)
    if p80: ax2.axvline(p80, color='#E76F51', lw=2, ls='--', alpha=0.8)

    ax2.set_yticks(ys)
    ax2.set_yticklabels(labels, fontsize=7)
    ax2.set_xlabel('Actual Duration (hours)')
    ax2.set_title('Analog Records')

    # Confidence tier badge
    ax1.text(0.97, 0.97, tier.replace('_', ' ').upper(),
             transform=ax1.transAxes, ha='right', va='top', fontsize=9,
             color='white', fontweight='bold',
             bbox=dict(facecolor=tier_color, edgecolor='none', pad=4, alpha=0.9))

    plt.tight_layout()
    plt.show()

    print(f"Retrieval summary: {summary}")

plot_analog_distribution(r1['analogs'], title='Stage D — RCP Seal Analog Duration Distribution')

### 3.4 · Stage E — Schedule Impact (Float Analysis)

**What Stage E does:**
Stage E quantifies the schedule impact of inserting the new activity into the
outage critical path. It uses a CPM (Critical Path Method) engine to compute:

- **Float**: how much scheduling slack exists at the insertion point
- **CP drag**: how many hours the activity would add to the total outage duration
- **Remaining float after insertion**: negative = critical path extended

> **Note:** In this demo, Stage E output is pre-built (stub) because the LOGOS
> CPM scheduling engine is not available in the demo environment.
> In production, Stage E calls the LOGOS API to compute these values in real-time.

**Key outputs to look for:** `remaining_float_after_hours` (negative = CP extension),
`cp_drag_hours` (directly added to outage end date)


In [ ]:
def plot_schedule_impact(schedule, title='Stage E — Schedule Impact Assessment'):
    float_analysis = schedule.get('float_analysis', {})
    cp_impact      = schedule.get('cp_impact', {})
    dur_est        = schedule.get('duration_estimate', {})
    insertion_pt   = schedule.get('insertion_point', {})

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(title, fontsize=12, fontweight='bold')

    # ── Panel 1: Float waterfall ──────────────────────────────────────────────
    avail  = float_analysis.get('available_float_before', 0)
    consumed = float_analysis.get('float_consumed_hours', 0)
    remaining = float_analysis.get('remaining_float_after', 0)
    criticality = float_analysis.get('criticality_label', 'unknown')

    bar_labels  = ['Float\nAvailable', 'Float\nConsumed', 'Float\nRemaining']
    bar_values  = [avail, consumed, max(remaining, 0)]
    bar_colors  = [
        PALETTE['proceed'],
        PALETTE['escalate'] if consumed > avail else PALETTE['defer'],
        PALETTE['proceed'] if remaining > 8 else
        (PALETTE['defer'] if remaining > 0 else PALETTE['escalate'])
    ]

    bars = ax1.bar(bar_labels, bar_values, color=bar_colors,
                   edgecolor='white', width=0.5)
    ax1.set_ylabel('Hours')
    ax1.set_title(f'Float Analysis ({criticality.upper()})')
    for bar, val in zip(bars, bar_values):
        ax1.text(bar.get_x() + bar.get_width()/2, val + 0.5,
                 f'{val:.0f} h', ha='center', fontsize=9, fontweight='bold')

    # Show negative remaining as annotation if applicable
    if remaining < 0:
        ax1.text(2, 1, f'({remaining:.0f} h deficit)',
                 ha='center', fontsize=8, color=PALETTE['escalate'])

    # ── Panel 2: CP impact + duration percentiles ─────────────────────────────
    cp_drag   = cp_impact.get('cp_drag_hours', 0)
    baseline  = cp_impact.get('baseline_cp_hours', 480)
    sens      = cp_impact.get('cp_sensitivity_score', 0)
    p50       = dur_est.get('p50_hours', 0)
    p80       = dur_est.get('p80_hours', 0)
    p90       = dur_est.get('p90_hours', 0)

    categories = ['p50 Duration', 'p80 Duration', 'p90 Duration', 'CP Drag']
    values     = [p50 or 0, p80 or 0, p90 or 0, cp_drag or 0]
    colors_v   = [PALETTE['stage_live']]*3 + [PALETTE['escalate'] if cp_drag > 24 else PALETTE['defer']]

    ax2.barh(categories, values, color=colors_v, edgecolor='white', height=0.5)
    ax2.set_xlabel('Hours')
    ax2.set_title(f'Duration Estimates & CP Drag\n'
                  f'Baseline CP: {baseline:.0f} h  |  Sensitivity: {sens:.2f}')
    ax2.axvline(24, color='#888', lw=1, ls=':', label='Escalate threshold (24 h)')
    for i, v in enumerate(values):
        ax2.text(v + 0.5, i, f'{v:.0f} h', va='center', fontsize=9)
    ax2.legend(fontsize=7)

    plt.tight_layout()
    plt.show()

    print(f"Insertion point : {insertion_pt.get('task_name', '?')} (task {insertion_pt.get('task_id', '?')})")
    print(f"Displaced tasks : {len(schedule.get('displaced_tasks', []))}  "
          f"(regulatory: {sum(1 for t in schedule.get('displaced_tasks',[]) if t.get('has_regulatory_constraint'))})")
    print(f"Resource conflicts: {len(schedule.get('resource_conflicts', []))}")
    for c in schedule.get('resource_conflicts', []):
        print(f"  [{c['conflict_type']:25s}] skill={c.get('skill_required')}")

plot_schedule_impact(r1['schedule'], title='Stage E — RCP Seal Schedule Impact')

### 3.5 · Stage F — Insertion Option Risk Scoring

**What Stage F does:**
Stage F generates every viable way to handle the new activity and scores
each option's risk using a multi-factor formula:

```
risk = 0.40 × cp_impact  +  0.30 × (1 − confidence)  +  0.20 × resource_score  +  0.10 × urgency
```

**Option types considered:**
| Option | Description |
|--------|-------------|
| `insert_now` | Insert immediately at the best available slot |
| `parallel` | Run concurrently with a non-critical task |
| `defer` | Defer to a later outage (only if regulatory permits) |
| `contingency_buffer` | Use schedule reserve time |
| `escalate` | Escalate to management (triggered when CP drag > 24 h) |

Options can be **infeasible** (crew conflict, critical task displacement) or
**regulatory-cleared = False** (TS constraint prohibits deferral).
The option with the **lowest risk score** among feasible, cleared options wins.

**Key output to look for:** `recommended_option_id` — the winning option


In [ ]:
def plot_option_risk_scores(options_result, title='Stage F — Insertion Option Risk Scores'):
    options   = options_result.get('options', [])
    rec_id    = options_result.get('recommended_option_id')
    summary   = options_result.get('ranking_summary', {})

    if not options:
        print('No options generated.')
        return

    fig, ax = plt.subplots(figsize=(11, max(3, len(options) * 0.8 + 1.5)))
    ax.set_title(title, fontsize=12, fontweight='bold')

    def _bar_color(opt):
        if opt.get('option_id') == rec_id:
            return PALETTE['proceed'] if opt.get('option_type') not in ('escalate_to_management',) \
                   else PALETTE['escalate']
        if not opt.get('feasible', True):
            return PALETTE['infeasible']
        if not opt.get('regulatory_cleared', True):
            return PALETTE['blocked']
        return '#A8DADC'

    labels = []
    scores = []
    colors = []
    hatches = []

    for opt in options:
        otype = opt.get('option_type', '?').replace('_', ' ')
        feasible = opt.get('feasible', True)
        cleared  = opt.get('regulatory_cleared', True)

        suffix = ''
        if opt.get('option_id') == rec_id:
            suffix = '  ← RECOMMENDED'
        elif not feasible:
            suffix = '  [INFEASIBLE]'
        elif not cleared:
            suffix = '  [REG BLOCKED]'

        labels.append(f"{otype}{suffix}")
        scores.append(opt.get('risk_score', 0))
        colors.append(_bar_color(opt))
        hatches.append('//' if not feasible or not cleared else '')

    y_pos = list(range(len(labels)))
    bars = ax.barh(y_pos, scores, color=colors, edgecolor='white',
                   height=0.55, linewidth=0.8)

    for bar, hatch in zip(bars, hatches):
        if hatch:
            bar.set_hatch(hatch)
            bar.set_edgecolor('#888')

    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel('Risk Score (lower = better)', fontsize=10)
    ax.set_xlim(0, 1.0)
    ax.axvline(0.5, color='#888', lw=1, ls=':', alpha=0.7)

    for i, (score, label) in enumerate(zip(scores, labels)):
        ax.text(score + 0.01, i, f'{score:.3f}', va='center', fontsize=8)

    ax.invert_yaxis()

    # Legend
    legend_items = [
        mpatches.Patch(color=PALETTE['proceed'],    label='Recommended (PROCEED)'),
        mpatches.Patch(color=PALETTE['escalate'],   label='Recommended (ESCALATE)'),
        mpatches.Patch(color='#A8DADC',             label='Feasible + cleared'),
        mpatches.Patch(color=PALETTE['blocked'],    label='Regulatory blocked'),
        mpatches.Patch(color=PALETTE['infeasible'], label='Infeasible'),
    ]
    ax.legend(handles=legend_items, fontsize=7, loc='lower right')

    plt.tight_layout()
    plt.show()

    print(f"Options summary: generated={summary.get('options_generated')} "
          f"feasible={summary.get('feasible_count')} "
          f"cleared={summary.get('regulatory_cleared_count')} "
          f"blocked={summary.get('regulatory_blocked_count')} "
          f"infeasible={summary.get('infeasible_count')} "
          f"best_score={summary.get('best_risk_score')}")

plot_option_risk_scores(r1['options'], title='Stage F — RCP Seal Option Risk Scores')

### 3.6 · Stage G — Recommendation Card

**What Stage G does:**
Stage G is the final synthesis stage. It takes all upstream outputs and produces:

1. A **decision status**: ESCALATE / PROCEED / DEFER / MONITOR / INCONCLUSIVE
2. An **executive summary** with the primary conclusion in plain language
3. An **evidence chain**: every data point that informed the decision (traceable)
4. An **analyst review** flag: if True, a human must review before action is taken

**For managers:** The recommendation card is the primary deliverable of the pipeline.
Everything above it is the computational evidence trail.


In [ ]:
def print_recommendation_card(result):
    """Print a formatted text recommendation card for one scenario result.

    Uses emoji status icons instead of ANSI terminal colour codes so the
    output renders correctly in Jupyter notebooks and exported HTML/PDF.
    """
    rec   = result['recommendation']
    summ  = rec.get('executive_summary', {})
    prim  = rec.get('primary_recommendation', {})
    hist  = rec.get('history_summary', {})
    sched = rec.get('schedule_summary', {})
    rev   = rec.get('analyst_review', {})
    flags = rec.get('attention_flags', [])
    reg   = rec.get('regulatory_flags', [])

    status = rec['decision_status']

    # Emoji icons render in Jupyter; ANSI escape codes do not
    STATUS_ICONS = {
        'ESCALATE':     '🔴',
        'PROCEED':      '🟢',
        'DEFER':        '🟡',
        'MONITOR':      '🔵',
        'INCONCLUSIVE': '⚪',
    }
    icon = STATUS_ICONS.get(status, '❓')

    width = 72
    bar   = '═' * width
    print(f'\n{bar}')
    print(f'  DACKAR RECOMMENDATION  —  {result["scenario_label"]}')
    print(bar)
    # Show icon alongside decision status for at-a-glance reading
    print(f'  DECISION          : {icon} {status}')
    print(f'  Confidence tier   : {summ.get("confidence_tier", "?")}')
    print(f'  Analyst review    : {rev.get("required", False)}')
    if rev.get('required') and rev.get('reason'):
        print(f'  Review reason     : {rev["reason"]}')
    print()
    print(f'  Primary conclusion:')
    conclusion = summ.get('primary_conclusion', 'N/A')
    import textwrap
    for line in textwrap.wrap(conclusion, width=66):
        print(f'    {line}')
    print()
    if prim:
        print(f'  Recommended option: {prim.get("option_type", "?")}')
        print(f'  CP impact (hours) : {prim.get("cp_impact_hours", 0):.1f} h')
        rationale = prim.get('rationale', '')
        for line in textwrap.wrap(rationale, width=66):
            print(f'    {line}')
    print()
    if flags:
        print('  Attention flags:')
        for f in flags:
            print(f'    ⚑ {f}')
        print()
    if reg:
        print(f'  Regulatory constraints ({len(reg)}):')
        for d in reg:
            print(f'    [{d["driver_type"]:35s}] defer_prohibited={d["defer_prohibited"]}')
        print()
    if reg_warn := summ.get('regulatory_warning'):
        print(f'  Regulatory warning : {reg_warn}')
    print(f'  Analog count      : {hist.get("analog_count", 0)} events across '
          f'{hist.get("outages_represented", 0)} outages')
    print(f'  Median duration   : {hist.get("median_actual_hours", "?")}')
    print()
    print(f'  Evidence chain    : {len(rec.get("evidence_chain", []))} items')
    for ev in rec.get('evidence_chain', [])[:4]:
        print(f'    [{ev.get("source_type","?")}] {ev.get("snippet","")[:60]}')
    print(bar)

# Print the card for Scenario 1 (RCP Seal — expected ESCALATE)
print_recommendation_card(r1)


In [ ]:
def plot_recommendation_card(result):
    """Render the Stage G recommendation as a matplotlib figure for presentations."""
    rec    = result['recommendation']
    summ   = rec.get('executive_summary', {})
    prim   = rec.get('primary_recommendation', {})
    flags  = rec.get('attention_flags', [])
    reg    = rec.get('regulatory_flags', [])
    hist   = rec.get('history_summary', {})
    rev    = rec.get('analyst_review', {})
    status = rec['decision_status']
    label  = result['scenario_label']

    STATUS_BG = {
        'ESCALATE':     '#D7263D',
        'PROCEED':      '#06A77D',
        'DEFER':        '#F4A261',
        'MONITOR':      '#A8DADC',
        'INCONCLUSIVE': '#94A3B8',
    }
    bg = STATUS_BG.get(status, '#888888')

    fig, ax = plt.subplots(figsize=(11, 6))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.axis('off')

    # ── Header band ───────────────────────────────────────────────────────────
    header = mpatches.FancyBboxPatch((0, 8.5), 10, 1.5, boxstyle='square',
                                      facecolor=bg, edgecolor='none')
    ax.add_patch(header)
    ax.text(5, 9.5, f'DACKAR — {label}', ha='center', va='center',
            fontsize=11, color='white', fontweight='bold')
    ax.text(5, 8.85, f'DECISION: {status}', ha='center', va='center',
            fontsize=18, color='white', fontweight='bold')

    # ── Body background ───────────────────────────────────────────────────────
    body = mpatches.FancyBboxPatch((0, 0), 10, 8.5, boxstyle='square',
                                    facecolor='#F8FAFC', edgecolor='#CBD5E1', linewidth=1)
    ax.add_patch(body)

    # ── Confidence tier pill ──────────────────────────────────────────────────
    tier = summ.get('confidence_tier', '?')
    tc   = PALETTE.get(tier, '#888')
    ax.text(0.3, 8.15, f'Confidence: {tier.replace("_"," ").upper()}',
            fontsize=8.5, color='white', fontweight='bold',
            bbox=dict(facecolor=tc, edgecolor='none', pad=4, boxstyle='round'))

    # ── Analyst review flag ───────────────────────────────────────────────────
    if rev.get('required'):
        ax.text(9.7, 8.15, '⚑ ANALYST REVIEW REQUIRED',
                fontsize=8, color=PALETTE['escalate'], fontweight='bold', ha='right')

    # ── Primary conclusion ────────────────────────────────────────────────────
    conclusion = summ.get('primary_conclusion', '')
    import textwrap
    wrapped = textwrap.fill(conclusion, 80)
    ax.text(0.3, 7.7, 'Primary Conclusion:', fontsize=9, fontweight='bold', color='#334155')
    ax.text(0.3, 7.3, wrapped, fontsize=8.5, color='#475569',
            wrap=True, va='top',
            bbox=dict(facecolor='#E2E8F0', edgecolor='none', pad=6, boxstyle='round'))

    # ── Recommended option ────────────────────────────────────────────────────
    opt_type = prim.get('option_type', '?').replace('_', ' ')
    opt_cp   = prim.get('cp_impact_hours', 0)
    opt_rat  = textwrap.fill(prim.get('rationale', ''), 80)
    ax.text(0.3, 5.8, f'Recommended option: {opt_type.upper()}  |  CP impact: {opt_cp:.0f} h',
            fontsize=9, fontweight='bold', color='#334155')
    ax.text(0.3, 5.5, opt_rat, fontsize=8, color='#475569', va='top')

    # ── Key metrics row ───────────────────────────────────────────────────────
    ax.text(0.3, 3.9, 'Key Metrics', fontsize=9, fontweight='bold', color='#334155')
    metrics = [
        ('Analogs',   f"{hist.get('analog_count', 0)} events / {hist.get('outages_represented', 0)} outages"),
        ('Median dur',f"{hist.get('median_actual_hours', '?')} h"),
        ('CP drag',   f"{prim.get('cp_impact_hours', 0):.0f} h"),
        ('Reg flags', str(len(reg))),
    ]
    for i, (k, v) in enumerate(metrics):
        xoff = 0.3 + i * 2.4
        box = mpatches.FancyBboxPatch((xoff, 3.0), 2.0, 0.75,
                                       boxstyle='round,pad=0.05',
                                       facecolor=bg, edgecolor='none', alpha=0.15)
        ax.add_patch(box)
        ax.text(xoff + 1.0, 3.55, k, ha='center', fontsize=7.5, color='#475569')
        ax.text(xoff + 1.0, 3.15, v, ha='center', fontsize=9, fontweight='bold', color='#1E293B')

    # ── Attention flags ───────────────────────────────────────────────────────
    ax.text(0.3, 2.7, 'Attention Flags', fontsize=9, fontweight='bold', color='#334155')
    if flags:
        for i, f in enumerate(flags[:4]):
            col = PALETTE['escalate'] if 'regulatory' in f or 'critical' in f else '#475569'
            ax.text(0.5, 2.35 - i*0.35, f'⚑  {f.replace("_", " ")}',
                    fontsize=8, color=col)
    else:
        ax.text(0.5, 2.35, 'No attention flags raised.', fontsize=8, color='#888')

    # ── Footer ────────────────────────────────────────────────────────────────
    ax.text(9.7, 0.15, f'Run ID: {result["run_id"]}',
            ha='right', fontsize=7, color='#94A3B8')

    plt.tight_layout()
    plt.show()

plot_recommendation_card(r1)

---
## 4 · Scenario 2 — Snubber Scope Expansion  🟢 PROCEED

**What happened?** While performing routine snubber inspections, engineers identified
additional snubbers requiring evaluation — a scope expansion common in aging plants.

**Why PROCEED?** This activity is:
- **Non-safety-related** (no TS constraint, deferral is technically allowed)
- **No critical-path impact** (0 h CP drag, 28 h of remaining float)
- **Well-supported by history** (5 analogous past activities → `data_supported` tier)

The pipeline recommends immediate insertion with high confidence.

Compare the cells below with Scenario 1 to see how each stage produces a
completely different evidence trail and conclusion from similar inputs.


In [ ]:
# Retrieve and display Stage A output for Scenario 2 (Snubber)
print('=== STAGE A — Snubber Scenario ===')
intake2 = r2['intake']

# Compare with Scenario 1: different emergence_type, no regulatory constraint
print(f"  Emergence type   : {intake2['emergence_type']}  (confidence {intake2['emergence_type_confidence']:.0%})")
print(f"  Has regulatory   : {intake2['has_regulatory_constraint']}")  # False — no TS constraint
print(f"  Data quality     : {intake2['data_quality_score']:.2f}")

# Reuse the same plot function defined for Scenario 1
plot_stage_a_summary(intake2, title=f"Stage A — {r2['scenario_label']}")


In [ ]:
# Stage C for Scenario 2 — Allen relations for snubber component history
# Expect: fewer/weaker prior events than the safety-critical RCP pump
plot_allen_timeline(r2['temporal'], title='Stage C — Snubber Scenario (Allen Relations)')


In [ ]:
# Stage D for Scenario 2 — duration distribution from 5 snubber analogue activities
# With 5 analogues, confidence_tier = 'data_supported' (threshold is ≥5)
# This is key: a higher confidence tier means insert_now beats defer on risk score
plot_analog_distribution(r2['analogs'], title='Stage D — Snubber Analog Duration Distribution')


In [ ]:
# Stage E for Scenario 2 — non-critical activity, zero CP drag
# remaining_float_after_hours = +28.0 (plenty of slack after insertion)
# Compare with Scenario 1 where remaining_float_after_hours = -44.0
plot_schedule_impact(r2['schedule'], title='Stage E — Snubber Schedule Impact (Non-Critical)')


In [ ]:
# Stage F for Scenario 2 — all options feasible, no regulatory constraint
# insert_now wins because: data_supported confidence (0.85) → low risk score
# defer is available but has higher risk due to urgency score
plot_option_risk_scores(r2['options'], title='Stage F — Snubber Option Risk Scores')


In [ ]:
# Stage G for Scenario 2 — print and plot the PROCEED recommendation card
# Note the 🟢 icon (vs 🔴 for Scenario 1) — the same pipeline, opposite conclusion
print_recommendation_card(r2)
plot_recommendation_card(r2)


---
## 5 · Side-by-Side Comparison

The chart below directly compares the two scenarios across five key pipeline metrics.
This is useful for show-and-tell to illustrate how the same pipeline adapts
its recommendation based on input characteristics.

**What to highlight for managers:**
- CP drag and regulatory constraint are the two dominant factors
- Analogue count determines confidence tier, which shifts the risk balance in Stage F
- The pipeline reaches opposite conclusions (ESCALATE vs PROCEED) with full traceability


In [ ]:
def plot_scenario_comparison(r1, r2):
    """Radar chart + bar comparison of key pipeline metrics for both scenarios."""
    import math

    fig = plt.figure(figsize=(15, 6))
    fig.suptitle('Scenario Comparison — DACKAR Pipeline Output', fontsize=13, fontweight='bold')

    # ── Radar chart (left) ────────────────────────────────────────────────────
    ax_radar = fig.add_subplot(1, 3, 1, polar=True)

    dimensions = [
        'Data Quality',
        'Analog\nCoverage',
        'Temporal\nCausality',
        'CP Float\nRemaining',
        'Option\nConfidence',
    ]
    N = len(dimensions)
    angles = [n / float(N) * 2 * math.pi for n in range(N)]
    angles += angles[:1]

    def _radar_values(r):
        rec     = r['recommendation']
        intake  = r['intake']
        analogs = r['analogs']
        sched   = r['schedule']
        options = r['options']
        temporal = r['temporal']

        dq = intake.get('data_quality_score', 0)

        analog_count = analogs.get('retrieval_summary', {}).get('analog_count', 0)
        analog_score = min(1.0, analog_count / 10.0)  # normalise to [0,1]

        strongest = temporal.get('summary', {}).get('strongest_link_score', 0) or 0

        # Float remaining normalised [0,1] — clamp negative to 0
        remaining = sched.get('float_analysis', {}).get('remaining_float_after', 0) or 0
        baseline  = sched.get('cp_impact', {}).get('baseline_cp_hours', 480) or 480
        float_score = max(0.0, min(1.0, remaining / baseline))

        # Recommended option confidence
        rec_id = options.get('recommended_option_id')
        opt_conf = 0.5
        for opt in options.get('options', []):
            if opt.get('option_id') == rec_id:
                opt_conf = float(opt.get('confidence') or 0.5)
                break

        return [dq, analog_score, strongest, float_score, opt_conf]

    vals1 = _radar_values(r1)
    vals2 = _radar_values(r2)
    vals1 += vals1[:1]
    vals2 += vals2[:1]

    ax_radar.set_xticks(angles[:-1])
    ax_radar.set_xticklabels(dimensions, fontsize=8)
    ax_radar.set_ylim(0, 1)
    ax_radar.set_yticks([0.25, 0.5, 0.75])
    ax_radar.set_yticklabels(['0.25', '0.5', '0.75'], fontsize=6)

    ax_radar.plot(angles, vals1, color=PALETTE['escalate'], lw=2, label=r1['scenario_label'][:20])
    ax_radar.fill(angles, vals1, color=PALETTE['escalate'], alpha=0.15)
    ax_radar.plot(angles, vals2, color=PALETTE['proceed'],  lw=2, label=r2['scenario_label'][:20])
    ax_radar.fill(angles, vals2, color=PALETTE['proceed'],  alpha=0.15)
    ax_radar.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=7)
    ax_radar.set_title('Risk Profile Radar', fontsize=10, pad=12)

    # ── Risk score comparison (middle) ────────────────────────────────────────
    ax_bar = fig.add_subplot(1, 3, 2)

    def _best_scores(r):
        opts = r['options'].get('options', [])
        rec_id = r['options'].get('recommended_option_id')
        feasible = [(o['option_type'].replace('_', ' '), o['risk_score'])
                    for o in opts
                    if o.get('feasible', True) and o.get('regulatory_cleared', True)]
        return feasible

    s1_scores = _best_scores(r1)
    s2_scores = _best_scores(r2)

    # Find common option types
    s1_dict = dict(s1_scores)
    s2_dict = dict(s2_scores)
    all_opts = sorted(set(list(s1_dict) + list(s2_dict)))

    x = np.arange(len(all_opts))
    w = 0.35
    ax_bar.bar(x - w/2, [s1_dict.get(o, 0) for o in all_opts],
               w, label='S1 ESCALATE', color=PALETTE['escalate'], alpha=0.8)
    ax_bar.bar(x + w/2, [s2_dict.get(o, 0) for o in all_opts],
               w, label='S2 PROCEED',  color=PALETTE['proceed'], alpha=0.8)
    ax_bar.set_xticks(x)
    ax_bar.set_xticklabels(all_opts, rotation=30, ha='right', fontsize=7.5)
    ax_bar.set_ylabel('Risk Score')
    ax_bar.set_title('Option Risk Scores\n(feasible + cleared only)')
    ax_bar.axhline(0.5, color='#888', lw=1, ls=':')
    ax_bar.legend(fontsize=8)
    ax_bar.set_ylim(0, 1.0)

    # ── Key metrics table (right) ─────────────────────────────────────────────
    ax_tbl = fig.add_subplot(1, 3, 3)
    ax_tbl.axis('off')
    ax_tbl.set_title('Key Metrics Comparison', fontsize=10)

    def _fmt(r):
        rec    = r['recommendation']
        intake = r['intake']
        sched  = r['schedule']
        analogs = r['analogs']
        summ   = rec.get('executive_summary', {})
        return [
            rec['decision_status'],
            intake.get('emergence_type', '?'),
            str(intake.get('has_regulatory_constraint', False)),
            sched.get('float_analysis', {}).get('criticality_label', '?'),
            f"{sched.get('cp_impact', {}).get('cp_drag_hours', 0):.0f} h",
            f"{analogs.get('retrieval_summary', {}).get('analog_count', 0)}",
            summ.get('confidence_tier', '?'),
            str(rec.get('analyst_review', {}).get('required', False)),
        ]

    rows = [
        'Decision',
        'Emergence type',
        'Regulatory',
        'Criticality',
        'CP drag',
        'Analog count',
        'Confidence',
        'Analyst review',
    ]
    col1 = _fmt(r1)
    col2 = _fmt(r2)

    table_data = [[r, c1, c2] for r, c1, c2 in zip(rows, col1, col2)]
    tbl = ax_tbl.table(
        cellText=table_data,
        colLabels=['Metric', 'S1 (RCP Seal)', 'S2 (Snubber)'],
        loc='center', cellLoc='left'
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8)
    tbl.scale(1, 1.6)

    # Colour decision rows
    for (row, col), cell in tbl.get_celld().items():
        if row == 0:
            cell.set_facecolor('#264653')
            cell.set_text_props(color='white', fontweight='bold')
        elif row == 1:  # Decision row
            if col == 1:
                cell.set_facecolor('#FECDD3')  # light red
            elif col == 2:
                cell.set_facecolor('#BBF7D0')  # light green
        elif row % 2 == 0:
            cell.set_facecolor('#F1F5F9')

    plt.tight_layout()
    plt.show()

plot_scenario_comparison(r1, r2)

---
## 6 · Evidence Chain Traceability

Every recommendation must cite its sources — this is a **trust architecture requirement** (§8).

The evidence chain lists every data item that informed the decision:
historical work orders, condition reports, schedule float data, and temporal events.
An analyst can click into any item to see the raw source record.

**For managers:** This is the audit trail. If the recommendation is ever questioned,
the evidence chain shows exactly what data drove the conclusion and how confident
the system was in each source.

**For developers:** The evidence chain is assembled in Stage G's
`_build_evidence_chain()` method, which pulls from all upstream stage outputs.


In [ ]:
def plot_evidence_chain(result, title='Stage G — Evidence Chain'):
    evidence = result['recommendation'].get('evidence_chain', [])
    if not evidence:
        print('No evidence chain items.')
        return

    SOURCE_COLORS = {
        'temporal_analysis':  PALETTE['stage_live'],
        'historical_analogs': '#2A9D8F',
        'schedule_analysis':  '#E76F51',
        'component_history':  '#F4A261',
        'regulatory':         PALETTE['escalate'],
    }

    fig, ax = plt.subplots(figsize=(13, max(3, len(evidence) * 0.8 + 1.5)))
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlim(0, 10)
    ax.set_ylim(-0.5, len(evidence) - 0.5)
    ax.axis('off')

    import textwrap
    for i, ev in enumerate(evidence):
        y = len(evidence) - 1 - i
        src   = ev.get('source_type', 'unknown')
        snip  = ev.get('snippet', '')
        conf  = ev.get('confidence', 0)
        color = SOURCE_COLORS.get(src, '#888')

        # Background band
        band = mpatches.FancyBboxPatch((0.1, y - 0.35), 9.8, 0.7,
                                        boxstyle='round,pad=0.05',
                                        facecolor='#F8FAFC', edgecolor='#CBD5E1', lw=0.5)
        ax.add_patch(band)

        # Source type badge
        ax.text(0.25, y, f'[{src.replace("_"," ").upper()}]',
                fontsize=7.5, va='center', color='white', fontweight='bold',
                bbox=dict(facecolor=color, edgecolor='none', pad=3, boxstyle='round'))

        # Snippet text
        ax.text(3.2, y, textwrap.shorten(snip, width=90, placeholder='…'),
                fontsize=8, va='center', color='#334155')

        # Confidence bar (right side)
        ax.barh(y, conf * 1.5, left=8.3, height=0.35,
                color=color, alpha=0.7)
        ax.text(9.85, y, f'{conf:.2f}', fontsize=7.5, va='center', color='#334155')

    ax.text(9.85, len(evidence) - 0.05, 'conf', fontsize=7, ha='center', color='#888')

    plt.tight_layout()
    plt.show()

print('=== Scenario 1 Evidence Chain ===')
plot_evidence_chain(r1, title='Stage G — RCP Seal Recommendation Evidence Chain')

print('\n=== Scenario 2 Evidence Chain ===')
plot_evidence_chain(r2, title='Stage G — Snubber Recommendation Evidence Chain')

---
## 7 · Trust Architecture Verification (§8 Requirements)

The `critical_analysis.md §8` trust architecture mandates that every recommendation
surface **four required fields** to enable analyst verification:

| Field | Purpose |
|-------|---------|
| (a) Outage IDs | Which historical outages contributed to the analogue pool |
| (b) Analog count | How many similar past activities were found |
| (c) Confidence tier | `data_supported` / `sme_informed` / `low_confidence` |
| (d) Rejection path | If the analyst rejects this recommendation, why and what was considered |

The cell below verifies that all four fields are present in both scenario outputs.


In [ ]:
# Verify that all four §8 trust-architecture fields are present in both outputs.
# This is a quick compliance check — in production, this would be a formal validator.
for label, r in [('Scenario 1 — RCP Seal', r1), ('Scenario 2 — Snubber', r2)]:
    rec   = r['recommendation']
    hist  = rec.get('history_summary', {})
    summ  = rec.get('executive_summary', {})
    rev   = rec.get('analyst_review', {})

    print(f'--- {label} ---')

    # (a) Outage IDs — the historical outages contributing analogues
    print(f'  (a) Outage IDs       : {hist.get("outage_ids", [])}')

    # (b) Analog count — how many similar past activities were retrieved
    print(f'  (b) Analog count     : {hist.get("analog_count", "?")}')

    # (c) Confidence tier — data_supported / sme_informed / low_confidence
    print(f'  (c) Confidence tier  : {summ.get("confidence_tier", "?")}')

    # (d) Rejection reason — None until an analyst explicitly rejects the recommendation;
    # we show a descriptive placeholder rather than bare 'None' for clarity
    rej = rev.get('rejection_reason')
    display = rej if rej else '(awaiting analyst action — not yet rejected)'
    print(f'  (d) Rejection reason : {display}')
    print()


---
## 8 · Raw Artifacts Inspector

Every stage output is a JSON-serialisable Python dictionary.
Change `STAGE_TO_INSPECT` below to explore any stage's full output structure.

**Available keys:** `intake`, `timeline`, `temporal`, `analogs`, `schedule`, `options`, `recommendation`

**For developers:** These raw artifacts are what would be stored to a database or
passed to downstream consumers (dashboard, notification service, audit log)
in a production deployment.


In [ ]:
import json

# Change these two values to inspect any stage's output for either scenario
STAGE_TO_INSPECT = 'recommendation'   # one of: intake, timeline, temporal, analogs,
                                      #         schedule, options, recommendation
SCENARIO         = r1                 # r1 = RCP Seal (ESCALATE)  |  r2 = Snubber (PROCEED)

# Pretty-print the selected artifact as JSON for easy inspection
artifact = SCENARIO.get(STAGE_TO_INSPECT, SCENARIO)
print(json.dumps(artifact, indent=2, default=str))
